# 09 — Predicting a Fixture

**Using the final model** · Ahamed M.A.U (IT24101779) · Group AI-39

Shows how the analytics staff would use the selected model (Logistic Regression, D-029) before a match:

1. Load the saved model (`models/final_logistic_regression.joblib`, created by `05_modeling.ipynb`).
2. Build the same leak-free features for a home team vs an away team from all matches played **before** the fixture.
3. Return the predicted outcome, its probabilities, and the factors that pushed the prediction.

A consistency check first confirms that this function reproduces the model's saved prediction for real test matches.

## 1. Setup

In [1]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.db import load_matches, load_teams
from src.preprocessing import add_match_outcome, sort_matches_chronologically
from src.feature_engineering import build_team_long, add_historical_features, to_match_level

bundle = joblib.load(REPO_ROOT / "models" / "final_logistic_regression.joblib")
model = bundle["model"]
FEATURES = bundle["feature_columns"]
FILL = bundle["imputation_values"]
print("Model trained on:", bundle["trained_on_seasons"][0], "to", bundle["trained_on_seasons"][-1],
      "| config:", bundle["config"], "| sklearn", bundle["sklearn_version"])

history = sort_matches_chronologically(add_match_outcome(load_matches()))
teams = load_teams().set_index("team_api_id")["team_long_name"]
team_ids = {name: tid for tid, name in teams.items()}
print("History:", len(history), "matches up to", history["date"].max().date())

Model trained on: 2009/2010 to 2014/2015 | config: {'C': 1.0, 'class_weight': 'balanced'} | sklearn 1.9.1


History: 25979 matches up to 2016-05-25


## 2. The prediction function

The fixture is added to the history as a future match with no score, and the project's own feature code
(`src/feature_engineering.py`) is run. Because every feature uses `shift(1)`, the fixture's features come only from
earlier matches, exactly as in training. Missing history (e.g. a newly promoted team) is filled with the
training-season medians stored with the model.

In [2]:
def fixture_features(home_id, away_id, date, league_id, history=history):
    """Leak-free features for one fixture, from matches strictly before `date`."""
    past = history[history["date"] < pd.Timestamp(date)]
    fixture = pd.DataFrame([{
        "id": -1, "date": pd.Timestamp(date), "season": "upcoming", "league_id": league_id,
        "home_team_api_id": home_id, "away_team_api_id": away_id,
        "home_team_goal": np.nan, "away_team_goal": np.nan,
    }])
    combined = pd.concat([past, fixture], ignore_index=True)
    feats = to_match_level(add_historical_features(build_team_long(combined))).set_index("id").loc[[-1]]
    filled = feats.isna().any(axis=None)
    for col, value in FILL.items():
        feats[col] = feats[col].fillna(value)
    feats["Form_Diff"] = feats["Home_Form_Last_5"] - feats["Away_Form_Last_5"]
    feats["Abs_Form_Diff"] = feats["Form_Diff"].abs()
    feats["league_id"] = league_id
    return feats[FEATURES], filled


def predict_fixture(home, away, date, history=history):
    """Predicted outcome, class probabilities and the three strongest drivers."""
    home_id, away_id = team_ids[home], team_ids[away]
    last_home = history[(history["home_team_api_id"] == home_id) | (history["away_team_api_id"] == home_id)]
    league_id = int(last_home.sort_values("date")["league_id"].iloc[-1])
    X, filled = fixture_features(home_id, away_id, date, league_id, history)

    proba = pd.Series(model.predict_proba(X)[0], index=model.classes_).reindex(["Home Win", "Draw", "Away Win"])
    predicted = proba.idxmax()

    # Logistic Regression explanation: contribution = coefficient x standardised feature value
    pre, clf = model.named_steps["preprocessor"], model.named_steps["classifier"]
    z = pd.Series(pre.transform(X)[0], index=pre.get_feature_names_out())
    coef = pd.Series(clf.coef_[list(clf.classes_).index(predicted)], index=z.index)
    contrib = (coef * z).drop([c for c in z.index if c.startswith("league_id")])
    drivers = contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(3)

    return {
        "fixture": f"{home} vs {away}", "date": str(pd.Timestamp(date).date()),
        "prediction": predicted, "probabilities": proba.round(3).to_dict(),
        "drivers": {k: round(v, 3) for k, v in drivers.items()},
        "features": X.iloc[0].round(3).to_dict(), "used_imputation": bool(filled),
    }

## 3. Consistency check against the saved test predictions

For real 2015/16 test matches, the function must reproduce the prediction stored in
`outputs/predictions/test_predictions.csv` by notebooks 05/06.

In [3]:
saved = pd.read_csv(REPO_ROOT / "outputs" / "predictions" / "test_predictions.csv", parse_dates=["date"])
sample = saved.sample(25, random_state=42)

agree = []
for row in sample.itertuples():
    m = history.loc[history["id"] == row.id].iloc[0]
    X, _ = fixture_features(m["home_team_api_id"], m["away_team_api_id"], m["date"], int(m["league_id"]))
    agree.append(model.predict(X)[0] == row.logistic_regression_prediction)
print(f"Function reproduces the saved prediction for {sum(agree)} of {len(agree)} random test matches")
assert all(agree)

Function reproduces the saved prediction for 25 of 25 random test matches


## 4. Example: predicting upcoming fixtures

Hypothetical fixtures dated just after the last season in the data, using every match up to the end of 2015/16. They
illustrate the output; they are not real fixtures from the database.

In [4]:
examples = [
    ("Arsenal", "Liverpool", "2016-08-14"),
    ("FC Barcelona", "Real Madrid CF", "2016-08-20"),
    ("Leicester City", "Aston Villa", "2016-08-13"),
]
results = [predict_fixture(h, a, d) for h, a, d in examples]
pd.DataFrame([{
    "fixture": r["fixture"], "prediction": r["prediction"],
    **{f"P({k})": v for k, v in r["probabilities"].items()},
    "top driver": next(iter(r["drivers"])),
} for r in results])

,fixture,prediction,P(Home Win),P(Draw),P(Away Win),top driver
0,Arsenal vs Liverpool,Home Win,0.434,0.302,0.264,Home_Win_Rate
1,FC Barcelona vs Real Madrid CF,Home Win,0.571,0.230,0.199,Home_Win_Rate
2,Leicester City vs Aston Villa,Home Win,0.511,0.284,0.205,Home_Avg_Goals_Last_5


In [5]:
# Full detail for one fixture: the inputs and what pushed the prediction
r = results[0]
print(r["fixture"], "→", r["prediction"], r["probabilities"])
print("Strongest drivers (positive = towards the predicted outcome):", r["drivers"])
pd.Series(r["features"], name="feature value").to_frame()

Arsenal vs Liverpool → Home Win {'Home Win': 0.434, 'Draw': 0.302, 'Away Win': 0.264}
Strongest drivers (positive = towards the predicted outcome): {'Home_Win_Rate': 0.275, 'Away_Win_Rate': -0.208, 'Home_Avg_Conceded_Last_5': 0.136}


,feature value
Home_Form_Last_5,11.000
Away_Form_Last_5,6.000
Home_Avg_Goals_Last_5,1.800
Away_Avg_Goals_Last_5,1.400
Home_Avg_Conceded_Last_5,0.400
Away_Avg_Conceded_Last_5,1.400
Home_Win_Rate,0.638
Away_Win_Rate,0.421
Form_Diff,5.000
Abs_Form_Diff,5.000


## 5. How to read the output

- **Prediction:** the most likely outcome. Use the **probabilities** to judge confidence: a fixture where no outcome
  is far ahead is a close call.
- **Drivers:** the features that moved the prediction most. Venue win rates usually dominate (D-030).
- **When to trust it:** the model is about 60% accurate when the form gap is large and no better than "home win" for
  evenly matched teams (report §8.3). Draws are rarely the top prediction.
- **Limits:** no injuries, line-ups or tactics; promoted teams fall back to training medians
  (`used_imputation = True`); probabilities are not calibrated. This is decision support, not a guarantee.